In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import sys
sys.path.append("..")
from src.preprocessing import df_to_densities
from src.forecasting import cv


import warnings
from scipy.integrate import IntegrationWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=IntegrationWarning)

Steps: <br>
1. Use cross-validation to select the best parameters for in-sample KDE <br>
2. Use cross-validation to select the number of dimensions for the dFPC using the resulting parameters for KDE in 1.

In [2]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df = pd.read_excel(returns_path, index_col="time")

# 1. Selecting KDE parameters

In [3]:
# Parameters to cross-validate
density_param_grid_0 = [
    {'kernel': ['gaussian', 'epanechnikov'],
        'bandwidth': ['silverman', 'scott', 'cv'],
        'adaptive': [True, False]}, 
    {'kernel': ['t_student'], 
        'df': range(2,6),
        'bandwidth': ['silverman', 'scott', 'cv'],
        'adaptive': [True, False]}]

density_param_grid = list(ParameterGrid(density_param_grid_0))

In [ ]:
params_dict = {}
records = []
for params in density_param_grid:
    print(params)
    df_support, df_densities = df_to_densities(df, params, verbose=False)
    try:
        cv_measures = cv(df_densities, df_support, initial_window=248)
    except Exception:
        continue
    for m in cv_measures:
        record = {
            # density
            "density_model": "f_hat",
            
            # parameters
            "kernel": params["kernel"],
            "bandwidth": params["bandwidth"],
            "adaptive": params["adaptive"],

            # CV info
            "fold": m["fold"],
            "method": m["method"],

            # metrics
            "KLD": float(m["KLD"]),
            "JSD": float(m["JSD"]),
            "L_1": float(m["L_1"]),
            "L_2": float(m["L_2"]),
            "L_INFTY": float(m["L_INFTY"]),
        }

        records.append(record)


results_df_1 = pd.DataFrame(records)

{'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'gaussian'}
>>> cv 1/1
{'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'epanechnikov'}
>>> cv 1/1
{'adaptive': True, 'bandwidth': 'scott', 'kernel': 'gaussian'}
>>> cv 1/1
{'adaptive': True, 'bandwidth': 'scott', 'kernel': 'epanechnikov'}
>>> cv 1/1
{'adaptive': True, 'bandwidth': 'cv', 'kernel': 'gaussian'}


In [ ]:
dimensions = np.arange(2,9)

array([2, 3, 4, 5, 6, 7, 8])